In [ ]:
import openpyxl
import requests
import time
import json

# Send request to get answer
def send_request(question):
    url = 'http://localhost:6006/api/local_doc_qa/local_doc_chat'
    headers = {
        'content-type': 'application/json'
    }
    data = {
        "user_id": "zzp",
        "kb_ids": ["KB0db68cee77644035ba25855a9527b301"],
        "question": question,
        "rerank": "True",  # whether to rerank
    }
    try:
        start_time = time.time()
        response = requests.post(url=url, headers=headers, json=data, timeout=60)
        end_time = time.time()
        res = response.json()
        # Print response status code and response time
        print(f"Response status code: {response.status_code}, response time: {end_time - start_time} seconds")
        # Return raw data
        return res.get('response', 'No answer returned')
    except Exception as e:
        print(f"Request failed: {e}")
        return 'Request failed'

# Read questions from Excel file
def read_excel(input_file):
    workbook = openpyxl.load_workbook(input_file)
    sheet = workbook.active
    questions = []

    # Assume questions are in column A, starting from the second row
    for row in sheet.iter_rows(min_row=2, max_col=1, values_only=True):
        questions.append(row[0])

    return questions

# Clean answer data
def clean_answers(answers):
    cleaned_answers = []
    for answer in answers:
        # If the answer is a JSON string, try to parse it and extract the answer field
        if isinstance(answer, str) and answer.startswith('data: {'):
            try:
                # Remove the leading "data: " and parse JSON
                json_data = json.loads(answer[6:])
                if 'answer' in json_data:
                    cleaned_answers.append(json_data['answer'])
                else:
                    cleaned_answers.append(answer)  # If no answer field, keep original data
            except json.JSONDecodeError:
                cleaned_answers.append(answer)  # If parsing fails, keep original data
        else:
            cleaned_answers.append(answer)  # If not a JSON string, keep original data
    return cleaned_answers

# Write to a new Excel file storing questions and answers
def write_to_new_excel(output_file, questions, answers, start_row=2):
    workbook = openpyxl.load_workbook(output_file) if start_row > 2 else openpyxl.Workbook()
    sheet = workbook.active
    sheet.title = "Questions and Answers"

    # Add header row if it's a new file
    if start_row == 2:
        sheet.cell(row=1, column=1, value="Question")
        sheet.cell(row=1, column=2, value="Answer")

    # Write questions and answers
    for row_idx, (question, answer) in enumerate(zip(questions, answers), start=start_row):
        sheet.cell(row=row_idx, column=1, value=question)
        sheet.cell(row=row_idx, column=2, value=answer)

    # Save the new file
    workbook.save(output_file)
    print(f"File saved to: {output_file} (rows {start_row}-{start_row+len(questions)-1})")

# Main function
def main():
    input_file = "files/QA_dataset.xlsx"  # Input Excel file path
    output_file = "files/QA_qwen7b_rerank_rep1.xlsx"  # Output Excel file path

    # Read questions from Excel file
    questions = read_excel(input_file)

    # Store questions and answers
    answers = []

    batch_size = 5  # Write every 5 questions and answers

    # Process questions and get answers
    for idx, question in enumerate(questions, start=1):
        print(f"Processing question {idx}: {question}")
        answer = send_request(question)
        answers.append(answer)

        # Write to file every batch_size questions
        if idx % batch_size == 0 or idx == len(questions):
            cleaned_answers = clean_answers(answers[-batch_size:])  # Clean the last batch_size answers
            write_to_new_excel(output_file, questions[idx - batch_size: idx], cleaned_answers, start_row=idx - batch_size + 2)

    print("All questions and answers processed!")

if __name__ == "__main__":
    main()

In [ ]:
import openpyxl
import requests
import time
import json

# Send request to get answer
def send_request(question):
    url = 'http://localhost:6006/api/local_doc_qa/local_doc_chat'
    headers = {
        'content-type': 'application/json'
    }
    data = {
        "user_id": "zzp",
        "kb_ids": ["KB0db68cee77644035ba25855a9527b301"],
        "rerank": "False",
    }
    try:
        start_time = time.time()
        response = requests.post(url=url, headers=headers, json=data, timeout=60)
        end_time = time.time()
        res = response.json()
        # Output response status code and response time
        print(f"Response status code: {response.status_code}, response time: {end_time - start_time} seconds")
        # Return raw data
        return res.get('response', 'No answer returned')
    except Exception as e:
        print(f"Request failed: {e}")
        return 'Request failed'

# Read questions from Excel file
def read_excel(input_file):
    workbook = openpyxl.load_workbook(input_file)
    sheet = workbook.active
    questions = []

    # Assume questions are in column A, starting from row 2
    for row in sheet.iter_rows(min_row=2, max_col=1, values_only=True):
        questions.append(row[0])

    return questions

# Clean answer data
def clean_answers(answers):
    cleaned_answers = []
    for answer in answers:
        # If answer is a JSON string, try to parse and extract the 'answer' field
        if isinstance(answer, str) and answer.startswith('data: {'):
            try:
                # Remove the leading "data: " and parse JSON
                json_data = json.loads(answer[6:])
                if 'answer' in json_data:
                    cleaned_answers.append(json_data['answer'])
                else:
                    cleaned_answers.append(answer)  # Keep original if no 'answer' field
            except json.JSONDecodeError:
                cleaned_answers.append(answer)  # Keep original if parsing fails
        else:
            cleaned_answers.append(answer)  # Keep original if not a JSON string
    return cleaned_answers

# Write questions and answers to a new Excel file
def write_to_new_excel(output_file, questions, answers, start_row=2):
    workbook = openpyxl.load_workbook(output_file) if start_row > 2 else openpyxl.Workbook()
    sheet = workbook.active
    sheet.title = "Questions and Answers"

    # Add header row if it's a new file
    if start_row == 2:
        sheet.cell(row=1, column=1, value="Question")
        sheet.cell(row=1, column=2, value="Answer")

    # Write questions and answers
    for row_idx, (question, answer) in enumerate(zip(questions, answers), start=start_row):
        sheet.cell(row=row_idx, column=1, value=question)
        sheet.cell(row=row_idx, column=2, value=answer)

    # Save the new file
    workbook.save(output_file)
    print(f"File saved to: {output_file} (rows {start_row}-{start_row+len(questions)-1})")

# Main function
def main():
    input_file = "files/QA_dataset.xlsx"  # Input Excel file path
    output_file = "files/QA_qwen7b_unrerank_rep1.xlsx"  # Output Excel file path

    # Read questions from Excel file
    questions = read_excel(input_file)

    # Store questions and answers
    answers = []

    batch_size = 5  # Write every 5 questions and answers

    # Process questions and get answers
    for idx, question in enumerate(questions, start=1):
        print(f"Processing question {idx}: {question}")
        answer = send_request(question)
        answers.append(answer)

        # Write to file after every batch_size questions
        if idx % batch_size == 0 or idx == len(questions):
            cleaned_answers = clean_answers(answers[-batch_size:])  # Clean the latest batch_size answers
            write_to_new_excel(output_file, questions[idx - batch_size: idx], cleaned_answers, start_row=idx - batch_size + 2)

    print("All questions and answers processed!")

if __name__ == "__main__":
    main()